In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for display (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Convert timestamp to timezone-aware datetime in Asia/Shanghai for any potential use
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) metric.csv aggregates per (cmdb_id, kpi_name)
metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_agg = (
    metric_grp
    .agg(
        count='count',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05),
        min='min',
        max='max'
    )
    .reset_index()
)
metric_top = metric_agg.sort_values('count', ascending=False).head(20)

# 2) trace.csv aggregates per (cmdb_id, trace_name)
trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_agg = (
    trace_grp
    .agg(
        count='count',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05),
        min='min',
        max='max'
    )
    .reset_index()
)
trace_top = trace_agg.sort_values('count', ascending=False).head(20)

# 3) log.csv aggregates per (cmdb_id, log_name)
log_grp = log_df.groupby(['cmdb_id', 'log_name'])['value']
log_agg = (
    log_grp
    .agg(
        count='count',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05),
        min='min',
        max='max'
    )
    .reset_index()
)
log_top = log_agg.sort_values('count', ascending=False).head(20)

# 4) error_logs.csv: distinct cmdb_id with counts and one sample message per cmdb_id
# Use first observed message as a sample; include count of messages
if 'message' in error_df.columns:
    error_agg = (
        error_df
        .groupby('cmdb_id')
        .agg(
            count=('message', 'count'),
            sample_message=('message', lambda x: x.iloc[0] if len(x) > 0 else None)
        )
        .reset_index()
        .sort_values('count', ascending=False)
    )
else:
    # fallback if no message column present
    error_agg = (
        error_df
        .groupby('cmdb_id')
        .size()
        .reset_index(name='count')
        .assign(sample_message=None)
        .sort_values('count', ascending=False)
    )

error_top = error_agg.head(20)

# Display compact results (top 20 rows each)
metric_top, trace_top, log_top, error_top

```
Out[1]:
```
Summary of the computed aggregates (top results only):

1) metric.csv (top pairs by sample count)
- Results are dominated by pairs with 31 samples each.
- Notable apache01 KPIs: Host Uptime ~6.56e6 (p95 ≈ 6.560003e6, p5 ≈ 6.558382e6), CPU idle/loads with p95/p5 in expected ranges (CPU idle ≈ 74% p95).
- Tomcat04 highlights: thread metrics (CurrentThreads values 100 and 16; MaxThreads = 500), request ErrorCount = 286 (constant), and a very large MaxTimeRequest value ≈ 324,852 for the sampled interval.
- Many Tomcat04 session metrics are zero across the sampled period.
- In short: apache01 and Tomcat04 produce the highest-count metric series; some KPIs show large maxima (uptime, request max time) and many zero-valued session metrics.

2) trace.csv (top pairs by sample count)
- IG01 and IG02 trace series each have 30 samples.
- Per-node latency summary: trace.self.duration_mean p95 ≈ 0.90 s (p5 ≈ 0.38 s), max ~1.7–1.8 s.
- trace.*.duration_p95 series have p95 values ≈ 4.1–4.4 s and p5 values around 1.0–2.0 s (these are the tail-percentile metrics).
- Row counts: trace.self.row_count p95 ≈ 2348 (IG01) and ≈ 2316 (IG02); per-Tomcat row_count p95 values ~288–304.
- In short: IG01/IG02 show consistent sampling; mean durations ~0.35–0.9s with p95 tail metrics around ~4s; request volumes (row_count) in the thousands.

3) log.csv (top pairs by sample count)
- Top log row counts: apache01 log.row_count p95 ≈ 2222 (count 31); Tomcat01 and Tomcat02 log.row_count p95 ≈ 610 and ≈ 583 respectively.
- log.error_count series are present for many components (apache01, Tomcat01/02/03/04, Mysql nodes) but error_count p95 = 0.0 in the sampled period (i.e., no nonzero aggregated error counts observed).
- GC pause maxima for Tomcat nodes are small (gc_pause_max p95 ≈ 0.09–0.13 s).
- In short: high log traffic on apache01; error_count is zero across these top series; small GC pause values on Tomcats.

4) error_logs.csv
- No error_log messages were returned in the aggregates (empty result). There are no sample messages/counts to report.

Overall takeaways:
- Data coverage is consistent: most key series for apache01, Tomcat nodes, and IG nodes have full sample counts (30–31).
- No aggregated log.error_count entries are nonzero in these top series; no error_log messages available to inspect.
- Trace tail-percentiles (duration_p95) are noticeably higher (~4s) while mean durations are sub-second (~0.3–0.9s).
- Some metric maxima (host uptime, a Tomcat request max time) stand out as large values and may merit further inspection if they are unexpected.

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count           p95            p5           min           max
1594  apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.560003e+06  6.558382e+06  6.558293e+06  6.560093e+06
1587  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  9.000000e-01  5.000000e-01  5.000000e-01  9.000000e-01
1528  apache01                        OSLinux-CPU_CPU_CPUidleutil     31  7.438932e+01  7.349722e+01  7.125330e+01  7.449970e+01
1524  apache01                            OSLinux-CPU_CPU_CPULoad     31  1.280000e+00  1.020000e+00  1.020000e+00  1.460000e+00
1525  apache01                         OSLinux-CPU_CPU_CPUSysTime     31  1.876890e+01  1.828347e+01  1.821202e+01  1.884240e+01
1526  apache01                        OSLinux-CPU_CPU_CPUUserTime     31  7.511839e+00  7.051375e+00  7.013500e+00  7.576800e+00
1527  apache01                             OSLinux-CPU_CPU_CPUWio     31  5.472500e-01  1.245000e-02  8.300000e-03  2.869800e+00
1512  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.000000e+02  1.000000e+02  1.000000e+02  1.000000e+02
1513  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  1.600000e+01  1.000000e+00  0.000000e+00  2.000000e+01
1514  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  5.000000e+02  5.000000e+02  5.000000e+02  5.000000e+02
1504  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1505  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1506  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1507  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1508  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1509  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1510  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1511  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1496  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  2.860000e+02  2.860000e+02  2.860000e+02  2.860000e+02
1497  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  3.248520e+05  3.248520e+05  3.248520e+05  3.248520e+05,    cmdb_id                       trace_name  count          p95           p5          min          max
0     IG01         trace.self.duration_mean     30     0.896757     0.378910     0.322075     1.702533
1     IG01          trace.self.duration_p95     30     4.186885     2.044600     1.285900    10.017000
2     IG01             trace.self.row_count     30  2348.300000  1439.300000  1330.000000  2664.000000
3     IG01  trace.to_Tomcat01.duration_mean     30     0.946150     0.362927     0.325661     1.518005
4     IG01   trace.to_Tomcat01.duration_p95     30     4.492370     1.418190     1.091250    10.016200
5     IG01      trace.to_Tomcat01.row_count     30   296.900000   172.050000   166.000000   327.000000
6     IG01  trace.to_Tomcat02.duration_mean     30     0.981595     0.391225     0.362172     1.684870
7     IG01   trace.to_Tomcat02.duration_p95     30     4.237695     2.049320     1.110200    10.015650
8     IG01      trace.to_Tomcat02.row_count     30   298.050000   184.350000   159.000000   336.000000
9     IG01  trace.to_Tomcat03.duration_mean     30     0.945713     0.357667     0.274124     1.657623
10    IG01   trace.to_Tomcat03.duration_p95     30     4.333710     1.702960     1.066000    10.014000
11    IG01      trace.to_Tomcat03.row_count     30   303.650000   180.750000   160.000000   330.000000
12    IG01  trace.to_Tomcat04.duration_mean     30     0.928859     0.391179     0.248270     1.840738
13    IG01   trace.to_Tomcat04.duration_p95     30     4.128125     2.047392     1.072350    10.012300
14    IG01      trace.to_Tomcat04.row_count     30   288.300000   175.450000   151.000000   339.000000
15    IG02         trace.self.duration_mean     30     0.923693     0.388412     0.301755     1.783884
16    IG02          trace.self.duration_p95     30     4.246950     2.053780     1.098000    10.016000
17    IG02             trace.self.row_count     30  2315.900000  1535.100000  1400.000000  2610.000000
18    IG02  trace.to_Tomcat01.duration_mean     30     0.936769     0.351400     0.331527     1.351722
19    IG02   trace.to_Tomcat01.duration_p95     30     4.366935     1.930405     1.454000     7.556500,      cmdb_id          log_name  count        p95        p5    min      max
9   Tomcat02     log.row_count     31   583.0000  345.5000  13.00   655.00
7   Tomcat02   log.error_count     31     0.0000    0.0000   0.00     0.00
6   Tomcat01     log.row_count     31   609.5000  362.0000  12.00   647.00
4   Tomcat01   log.error_count     31     0.0000    0.0000   0.00     0.00
17  apache01     log.row_count     31  2222.0000  887.0000  22.00  2360.00
16  apache01   log.error_count     31     0.0000    0.0000   0.00     0.00
13  Tomcat04   log.error_count     31     0.0000    0.0000   0.00     0.00
12  Tomcat03     log.row_count     31   583.0000  345.0000   7.00   680.00
10  Tomcat03   log.error_count     31     0.0000    0.0000   0.00     0.00
15  Tomcat04     log.row_count     31   562.5000  361.0000   9.00   647.00
8   Tomcat02  log.gc_pause_max     30     0.1265    0.0300   0.02     0.18
0    Mysql01   log.error_count     27     0.0000    0.0000   0.00     0.00
19  apache02     log.row_count     27  1030.2000   48.1000  15.00  1322.00
18  apache02   log.error_count     27     0.0000    0.0000   0.00     0.00
1    Mysql01     log.row_count     27    26.6000    1.0000   1.00    43.00
5   Tomcat01  log.gc_pause_max     24     0.0985    0.0315   0.03     0.10
11  Tomcat03  log.gc_pause_max     24     0.0900    0.0215   0.02     0.10
14  Tomcat04  log.gc_pause_max     24     0.0985    0.0415   0.03     0.10
3    Mysql02     log.row_count      1     1.0000    1.0000   1.00     1.00
2    Mysql02   log.error_count      1     0.0000    0.0000   0.00     0.00, Empty DataFrame
Columns: [cmdb_id, count, sample_message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing metric_df if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure timezone-aware datetime in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the full series BEFORE filtering
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# Merge thresholds back to the full metric dataframe
metric_with_thresh = metric_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Filter metric points to the incident duration in UTC+8: 2021-03-10 15:00 to 2021-03-10 15:30 (inclusive)
start = pd.Timestamp('2021-03-10 15:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-10 15:30:00').tz_localize(tz)
window_df = metric_with_thresh[(metric_with_thresh['dt'] >= start) & (metric_with_thresh['dt'] <= end)].copy()

# 3) Identify anomalies: value >= global P95 OR value <= global P5
# (global thresholds were computed on the full series above)
window_df['is_anomaly'] = (window_df['value'] >= window_df['p95']) | (window_df['value'] <= window_df['p5'])

anomalies = window_df[window_df['is_anomaly']].copy()

# Prepare compact outputs
# Output 1: Top 50 anomaly rows sorted by timestamp ascending with selected columns
anomalies_top50 = (
    anomalies
    .sort_values('timestamp', ascending=True)
    .loc[:, ['cmdb_id', 'kpi_name', 'timestamp', 'value', 'p95', 'p5']]
    .head(50)
)

# Output 2: Aggregated anomaly counts per cmdb_id (top 20)
anomaly_counts = (
    anomalies
    .groupby('cmdb_id')
    .size()
    .reset_index(name='anomaly_count')
    .sort_values('anomaly_count', ascending=False)
    .head(20)
)

anomalies_top50, anomaly_counts

```
Out[2]:
```
# Build a concise plain-English summary from the previously computed anomaly results
# Reuse in-memory variables from prior execution: anomalies, anomalies_top50, anomaly_counts
import pandas as pd, pytz
tz = pytz.timezone('Asia/Shanghai')

# Safeguard: if variables aren't present, inform user (but they should exist from prior run)
try:
    total_anomalies = int(anomalies.shape[0])
    top50_count = int(anomalies_top50.shape[0])
    earliest_ts = int(anomalies_top50['timestamp'].min()) if top50_count>0 else None
    earliest_dt = pd.to_datetime(earliest_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z') if earliest_ts else 'N/A'
    top_components_list = anomaly_counts.head(20).values.tolist()
except Exception as e:
    summary = f"Error: required anomaly variables not found in the environment: {e}"
else:
    # Format top components as "name(count)"
    comps_formatted = ', '.join([f"{row[0]}({row[1]})" for row in top_components_list])
    # Compose summary string
    summary = (
        f"Metric anomaly summary for the incident window (2021-03-10 15:00–15:30 UTC+8):\n"
        f"- Total anomalies detected in window: {total_anomalies}.\n"
        f"- Showing top {top50_count} anomaly rows (sample earliest timestamp in this sample: {earliest_dt}).\n"
        f"- Top components by anomaly count (descending, up to 20): {comps_formatted}.\n\n"
        f"Observations:\n"
        f"- Anomalies are heavily concentrated in database and caching components: Mysql01 and Mysql02 have the highest counts (~{anomaly_counts.iloc[0,1]} and {anomaly_counts.iloc[1,1]} respectively), followed by Redis02 and Redis01.\n"
        f"- Tomcat instances (Tomcat01–04) also show substantial anomaly counts (each ~1k+ for Tomcat nodes).\n"
        f"- The top-50 sample rows include filesystem and MySQL-related metrics at the incident timestamp (many entries at timestamp {earliest_ts} i.e. {earliest_dt}), and multiple zero-valued MySQL metric points flagged as anomalies (value == global P5 in those series).\n"
        f"- Actionable next steps: focus investigation on Mysql and Redis nodes first (highest anomaly counts), then Tomcat nodes and IG components; inspect the sampled metric types flagged (filesystem usage, MySQL buffer/pool metrics, request/max-time metrics) for root causes.\n"
    )

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name   timestamp        value        p95           p5
47      IG02           OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace  1615359600  4318.000000  4318.0000  4318.000000
48      IG02  OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace  1615359600   768.000000   768.0000   768.000000
49      IG02        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity  1615359600    15.416955    15.4170    15.416955
50      IG02  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...  1615359600     0.471497     0.4715     0.471497
51      IG02       OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace  1615359600   140.000000   140.0000   140.000000
..       ...                                                ...         ...          ...        ...          ...
210  Mysql01     Mysql-MySQL_3306_Innodb buffer pool pages misc  1615359600     0.000000     0.0000     0.000000
211  Mysql01    Mysql-MySQL_3306_Innodb buffer pool pages total  1615359600     0.000000     0.0000     0.000000
212  Mysql01     Mysql-MySQL_3306_Innodb buffer pool read ahead  1615359600     0.000000     0.0000     0.000000
188  Mysql01           Mysql-MySQL_3306_Created Tmp Disk Tables  1615359600     0.000000     0.0000     0.000000
189  Mysql01                 Mysql-MySQL_3306_Created Tmp Files  1615359600     0.000000     0.0000     0.000000

[50 rows x 6 columns],           cmdb_id  anomaly_count
4         Mysql01           2915
5         Mysql02           2794
7         Redis02           1552
6         Redis01           1526
22       Tomcat04           1145
20       Tomcat02           1136
21       Tomcat03           1056
19       Tomcat01           1022
0            IG01            822
1            IG02            813
2            MG01            800
24       apache02            797
23       apache01            779
3            MG02            753
12   ServiceTest3             70
10  ServiceTest11             70
9   ServiceTest10             70
8    ServiceTest1             70
11   ServiceTest2             70
14   ServiceTest5             70)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse anomalies from memory; raise clear error if not available
if 'anomalies' not in globals():
    raise RuntimeError("In-memory DataFrame 'anomalies' not found. Please run the prior anomaly detection step first.")

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp is integer type
anomalies['timestamp'] = anomalies['timestamp'].astype(int)

# Helper to compute max consecutive run length (in minutes) where adjacent timestamps differ by exactly 60s
def max_consecutive_run(ts_series):
    ts = np.sort(ts_series.values.astype(np.int64))
    if ts.size == 0:
        return 0
    if ts.size == 1:
        return 1
    diffs = np.diff(ts)
    # consecutive where diff == 60
    # identify run segments: start new run when diff != 60
    run_breaks = diffs != 60
    # run ids for each element except first; build full run ids array for elements
    run_ids = np.empty(ts.size, dtype=int)
    run_ids[0] = 0
    run_ids[1:] = np.cumsum(run_breaks)
    # count sizes per run id
    run_sizes = np.bincount(run_ids)
    return int(run_sizes.max())

# Group by (cmdb_id, kpi_name) and compute requested aggregates
grouped = (
    anomalies
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        anomaly_count=('timestamp', 'size'),
        earliest_anomaly_ts=('timestamp', 'min')
    )
    .reset_index()
)

# Compute earliest_anomaly_dt and max_consecutive_run per group
# We'll map earliest_anomaly_dt from earliest_anomaly_ts
grouped['earliest_anomaly_dt'] = pd.to_datetime(grouped['earliest_anomaly_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Compute max_consecutive_run by applying to each group
max_runs = (
    anomalies
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .apply(max_consecutive_run)
    .reset_index(name='max_consecutive_run')
)

# Merge max_consecutive_run into grouped
grouped = grouped.merge(max_runs, on=['cmdb_id', 'kpi_name'], how='left')

# Prepare Output A: Top 20 (cmdb_id, kpi_name) by anomaly_count desc
output_A = grouped.sort_values('anomaly_count', ascending=False).loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_ts', 'earliest_anomaly_dt', 'max_consecutive_run']].head(20)

# Output B: aggregated per-component summary
comp_summary = (
    anomalies
    .groupby('cmdb_id')['timestamp']
    .agg(total_anomaly_count='size', earliest_anomaly_ts='min')
    .reset_index()
)
comp_summary['earliest_anomaly_dt'] = pd.to_datetime(comp_summary['earliest_anomaly_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
output_B = comp_summary.sort_values('total_anomaly_count', ascending=False).loc[:, ['cmdb_id', 'total_anomaly_count', 'earliest_anomaly_ts', 'earliest_anomaly_dt']].head(20)

# Display compact outputs
output_A, output_B

```
Out[3]:
```
summary = (
    "Aggregate anomaly summary (component + KPI level):\n"
    "- Many KPI series (notably Tomcat session/thread/request metrics) had anomaly_count = 31 and max_consecutive_run = 31, "
    "meaning they were anomalous every minute across the entire incident window (2021-03-10 15:00–15:30 UTC+8).\n"
    "- All grouped earliest anomalies occur at 2021-03-10 15:00:00 (UTC+8).\n"
    "- By component, the highest anomaly volumes are: Mysql01 (2915 anomalies), Mysql02 (2794), Redis02 (1552), Redis01 (1526), "
    "followed by Tomcat04/02/03/01 (each ~1k+ anomalies). Other components (apache, MG, IG) show lower but nontrivial counts.\n"
    "- Interpretation / recommended focus: prioritize investigation on Mysql and Redis nodes first (largest anomaly counts), then Tomcat instances. "
    "The Tomcat KPI patterns (continuous 31-minute runs) suggest persistent, sustained deviations rather than single transient spikes.\n"
    "- Example notable pattern: multiple Tomcat session/thread KPIs have continuous anomalies across the full window, indicating a systemic issue affecting those metrics."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_ts      earliest_anomaly_dt  max_consecutive_run
1512  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31           1615359600  2021-03-10 15:00:00 CST                   31
1514  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31           1615359600  2021-03-10 15:00:00 CST                   31
1504  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31           1615359600  2021-03-10 15:00:00 CST                   31
1505  Tomcat04     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31           1615359600  2021-03-10 15:00:00 CST                   31
1506  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31           1615359600  2021-03-10 15:00:00 CST                   31
1507  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31           1615359600  2021-03-10 15:00:00 CST                   31
1508  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31           1615359600  2021-03-10 15:00:00 CST                   31
1509  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600  2021-03-10 15:00:00 CST                   31
1510  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600  2021-03-10 15:00:00 CST                   31
1511  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600  2021-03-10 15:00:00 CST                   31
1496  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31           1615359600  2021-03-10 15:00:00 CST                   31
1497  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31           1615359600  2021-03-10 15:00:00 CST                   31
1500  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31           1615359600  2021-03-10 15:00:00 CST                   31
1501  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31           1615359600  2021-03-10 15:00:00 CST                   31
1502  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31           1615359600  2021-03-10 15:00:00 CST                   31
1503  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter             31           1615359600  2021-03-10 15:00:00 CST                   31
1400  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600  2021-03-10 15:00:00 CST                   31
1401  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600  2021-03-10 15:00:00 CST                   31
1402  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600  2021-03-10 15:00:00 CST                   31
1403  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31           1615359600  2021-03-10 15:00:00 CST                   31,           cmdb_id  total_anomaly_count  earliest_anomaly_ts      earliest_anomaly_dt
4         Mysql01                 2915           1615359600  2021-03-10 15:00:00 CST
5         Mysql02                 2794           1615359600  2021-03-10 15:00:00 CST
7         Redis02                 1552           1615359600  2021-03-10 15:00:00 CST
6         Redis01                 1526           1615359600  2021-03-10 15:00:00 CST
22       Tomcat04                 1145           1615359600  2021-03-10 15:00:00 CST
20       Tomcat02                 1136           1615359600  2021-03-10 15:00:00 CST
21       Tomcat03                 1056           1615359600  2021-03-10 15:00:00 CST
19       Tomcat01                 1022           1615359600  2021-03-10 15:00:00 CST
0            IG01                  822           1615359600  2021-03-10 15:00:00 CST
1            IG02                  813           1615359600  2021-03-10 15:00:00 CST
2            MG01                  800           1615359600  2021-03-10 15:00:00 CST
24       apache02                  797           1615359600  2021-03-10 15:00:00 CST
23       apache01                  779           1615359600  2021-03-10 15:00:00 CST
3            MG02                  753           1615359600  2021-03-10 15:00:00 CST
12   ServiceTest3                   70           1615359600  2021-03-10 15:00:00 CST
10  ServiceTest11                   70           1615359600  2021-03-10 15:00:00 CST
9   ServiceTest10                   70           1615359600  2021-03-10 15:00:00 CST
8    ServiceTest1                   70           1615359600  2021-03-10 15:00:00 CST
11   ServiceTest2                   70           1615359600  2021-03-10 15:00:00 CST
14   ServiceTest5                   70           1615359600  2021-03-10 15:00:00 CST)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse in-memory 'anomalies' DataFrame
if 'anomalies' not in globals():
    raise RuntimeError("In-memory DataFrame 'anomalies' not found. Run the anomaly detection step first.")

tz = pytz.timezone('Asia/Shanghai')

# Ensure types
anomalies['timestamp'] = anomalies['timestamp'].astype(int)
# Ensure p95 and p5 columns exist (they were merged earlier)
if not {'p95', 'p5'}.issubset(anomalies.columns):
    raise RuntimeError("Expected columns 'p95' and 'p5' missing from anomalies DataFrame.")

# Top candidate components to focus on
top_components = ['Mysql01','Mysql02','Redis01','Redis02','Tomcat01','Tomcat02','Tomcat03','Tomcat04']

# Filter anomalies to these components
focus_anoms = anomalies[anomalies['cmdb_id'].isin(top_components)].copy()

# Define or reuse max_consecutive_run helper
def max_consecutive_run(ts_series):
    ts = np.sort(ts_series.values.astype(np.int64))
    if ts.size == 0:
        return 0
    if ts.size == 1:
        return 1
    diffs = np.diff(ts)
    run_breaks = diffs != 60
    run_ids = np.empty(ts.size, dtype=int)
    run_ids[0] = 0
    run_ids[1:] = np.cumsum(run_breaks)
    run_sizes = np.bincount(run_ids)
    return int(run_sizes.max())

# Group by (cmdb_id, kpi_name) and compute required aggregates
grouped = (
    focus_anoms
    .groupby(['cmdb_id', 'kpi_name'])
    .agg(
        anomaly_count=('timestamp', 'size'),
        earliest_anomaly_ts=('timestamp', 'min'),
        # compute counts for p5 equality and >= p95
        count_eq_p5=('value', lambda v: np.sum(np.isclose(v.values, focus_anoms.loc[v.index, 'p5'].values))),
        count_ge_p95=('value', lambda v: np.sum(v.values >= focus_anoms.loc[v.index, 'p95'].values)),
        min_value=('value', 'min'),
        median_value=('value', 'median'),
        max_value=('value', 'max')
    )
    .reset_index()
)

# Compute max_consecutive_run per group using apply on the timestamps
runs = (
    focus_anoms
    .groupby(['cmdb_id', 'kpi_name'])['timestamp']
    .apply(max_consecutive_run)
    .reset_index(name='max_consecutive_run')
)

grouped = grouped.merge(runs, on=['cmdb_id', 'kpi_name'], how='left')

# Compute percentages
grouped['pct_eq_p5'] = (grouped['count_eq_p5'] / grouped['anomaly_count']) * 100.0
grouped['pct_ge_p95'] = (grouped['count_ge_p95'] / grouped['anomaly_count']) * 100.0

# Convert earliest_anomaly_ts to Asia/Shanghai datetime string
grouped['earliest_anomaly_dt'] = pd.to_datetime(grouped['earliest_anomaly_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select and order columns for Output 1
per_kpi = grouped.loc[:, ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_ts', 'max_consecutive_run', 'pct_eq_p5', 'pct_ge_p95', 'min_value', 'median_value', 'max_value']]

# Sort by anomaly_count desc and limit to top 20 rows
per_kpi_top20 = per_kpi.sort_values('anomaly_count', ascending=False).head(20)

# Output 2: per-component summary for the same components
comp_grp = (
    focus_anoms
    .groupby('cmdb_id')
    .agg(
        total_anomaly_count=('timestamp', 'size'),
        earliest_anomaly_ts=('timestamp', 'min'),
        count_eq_p5=('value', lambda v: np.sum(np.isclose(v.values, focus_anoms.loc[v.index, 'p5'].values))),
        count_ge_p95=('value', lambda v: np.sum(v.values >= focus_anoms.loc[v.index, 'p95'].values))
    )
    .reset_index()
)

comp_grp['pct_component_rows_eq_p5'] = (comp_grp['count_eq_p5'] / comp_grp['total_anomaly_count']) * 100.0
comp_grp['pct_component_rows_ge_p95'] = (comp_grp['count_ge_p95'] / comp_grp['total_anomaly_count']) * 100.0
comp_grp['earliest_anomaly_dt'] = pd.to_datetime(comp_grp['earliest_anomaly_ts'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select and order columns for Output 2; ensure only the 8 components (they are already filtered)
comp_summary = comp_grp.loc[:, ['cmdb_id', 'total_anomaly_count', 'earliest_anomaly_ts', 'earliest_anomaly_dt', 'pct_component_rows_eq_p5', 'pct_component_rows_ge_p95']]
comp_summary = comp_summary.sort_values('total_anomaly_count', ascending=False)

# Display compact results (as variables)
per_kpi_top20, comp_summary

```
Out[4]:
```
Summary of focused anomaly analysis for the selected top components (Mysql01, Mysql02, Redis01, Redis02, Tomcat01–04):

- Scope and timing
  - All groups' earliest anomaly is at 2021-03-10 15:00:00 (UTC+8).
  - Many KPI groups were anomalous for the entire incident window: max_consecutive_run = 31 (i.e., 31 continuous minutes from 15:00 to 15:30).

- Per (component, KPI) highlights (top rows by anomaly_count)
  - Many Tomcat KPI groups (session/thread/request metrics) have anomaly_count = 31, max_consecutive_run = 31, and 100% of anomaly rows equal to p5 and also flagged >= p95. Example values show constants like 0.0 (session metrics), 100/500 (thread counts), 302 (error count), and a very large MaxTimeRequest (330,351).
  - Several Mysql02 OS/process/network KPIs (e.g., TCP CLOSE-WAIT, FIN-WAIT, process counts) show anomaly_count = 31 and values = 0.0 (p5 == observed value).
  - Most anomaly rows for many KPI groups are extreme values (either exactly the global P5 or >= global P95) rather than dispersed spikes.

- Per-component aggregates (these 8 components only)
  - Total anomaly counts (descending): Mysql01 2915, Mysql02 2794, Redis02 1552, Redis01 1526, Tomcat04 1145, Tomcat02 1136, Tomcat03 1056, Tomcat01 1022.
  - High fractions of anomalies are equal to the global P5: typically ~85–90% per component (Mysql01 ~88.7%, Mysql02 ~85.6%, Redis ~86–89%, Tomcats ~89–90%).
  - A similarly high fraction of anomalies are >= global P95 (roughly 83–90% across these components).

- Interpretation and recommended next steps
  - The anomaly patterns are persistent (continuous across the whole window) and concentrated in Mysql and Redis first, then Tomcat instances.
  - A large portion of anomalies are exactly at the computed global P5 (many zero-valued metrics) or at/above P95 — this indicates metrics are pegged to their historical extremes during the window (either suppressed to a low bound or hitting a high tail).
  - Investigative priorities: inspect Mysql01 and Mysql02 (highest anomaly volumes), then Redis nodes, then Tomcats. Pay attention to:
    - MySQL: buffer/pool, created-temp files/tables, network TCP state metrics that are zero or abnormal.
    - Redis: the high count of extreme values (look for evictions, memory, or network issues).
    - Tomcats: session/thread counters being zero or fixed, high ErrorCount and very large MaxTimeRequest values.
  - Because anomalies are continuous and start at the same timestamp, check any system-wide change or deployment that coincided with 2021-03-10 15:00 (configuration, deployment, network/CICD action, or scheduled job).

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_ts  max_consecutive_run   pct_eq_p5  pct_ge_p95  min_value  median_value  max_value
1096  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
1097  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
434    Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
435    Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
437    Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
440    Mysql02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...             31           1615359600                   31   12.903226   87.096774        0.5           0.6        0.6
441    Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31           1615359600                   31  100.000000  100.000000       24.0          24.0       24.0
987   Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
988   Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
989   Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
990   Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
991   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
981   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31           1615359600                   31  100.000000  100.000000      302.0         302.0      302.0
982   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31           1615359600                   31  100.000000  100.000000   330351.0      330351.0   330351.0
993   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
994   Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
995   Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
996   Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31           1615359600                   31  100.000000  100.000000        0.0           0.0        0.0
997   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31           1615359600                   31  100.000000  100.000000      100.0         100.0      100.0
999   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...             31           1615359600                   31  100.000000  100.000000      500.0         500.0      500.0,     cmdb_id  total_anomaly_count  earliest_anomaly_ts      earliest_anomaly_dt  pct_component_rows_eq_p5  pct_component_rows_ge_p95
0   Mysql01                 2915           1615359600  2021-03-10 15:00:00 CST                 88.747856                  87.512864
1   Mysql02                 2794           1615359600  2021-03-10 15:00:00 CST                 85.612026                  83.249821
3   Redis02                 1552           1615359600  2021-03-10 15:00:00 CST                 88.917526                  86.726804
2   Redis01                 1526           1615359600  2021-03-10 15:00:00 CST                 86.631717                  85.255570
7  Tomcat04                 1145           1615359600  2021-03-10 15:00:00 CST                 89.082969                  86.462882
5  Tomcat02                 1136           1615359600  2021-03-10 15:00:00 CST                 89.348592                  88.204225
6  Tomcat03                 1056           1615359600  2021-03-10 15:00:00 CST                 90.340909                  90.246212
4  Tomcat01                 1022           1615359600  2021-03-10 15:00:00 CST                 90.019569                  86.986301)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse trace_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp is int and create timezone-aware dt in UTC+8
trace_df['timestamp'] = trace_df['timestamp'].astype(int)
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series BEFORE filtering
thresholds = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# Merge thresholds back
trace_with_thresh = trace_df.merge(thresholds, on=['cmdb_id', 'trace_name'], how='left')

# 2) Filter to incident window (inclusive) in Asia/Shanghai: 2021-03-10 15:00 to 2021-03-10 15:30
start = pd.Timestamp('2021-03-10 15:00:00').tz_localize(tz)
end = pd.Timestamp('2021-03-10 15:30:00').tz_localize(tz)
window_df = trace_with_thresh[(trace_with_thresh['dt'] >= start) & (trace_with_thresh['dt'] <= end)].copy()

# 3) Mark anomalies: value >= p95 OR value <= p5
window_df['is_anomaly'] = (window_df['value'] >= window_df['p95']) | (window_df['value'] <= window_df['p5'])
anomalies = window_df[window_df['is_anomaly']].copy()

# Output 1: Top 20 anomaly rows sorted by timestamp ascending with selected columns
anomalies_top20 = anomalies.sort_values('timestamp', ascending=True).loc[:, ['cmdb_id', 'trace_name', 'timestamp', 'value', 'p95', 'p5']].head(20)

# Output 2: Aggregated anomaly counts per (trace_name, cmdb_id), sorted by count desc (top 20)
agg_by_tracename = (
    anomalies
    .groupby(['trace_name', 'cmdb_id'])
    .agg(anomaly_count=('timestamp', 'size'), earliest_anomaly_ts=('timestamp', 'min'))
    .reset_index()
    .sort_values('anomaly_count', ascending=False)
    .head(20)
)

# Focused summary for trace.to_Tomcat0[1-4] prefixes: one row per target prefix (aggregate across matching trace_name variants)
prefixes = ['trace.to_Tomcat01', 'trace.to_Tomcat02', 'trace.to_Tomcat03', 'trace.to_Tomcat04']
focused_rows = []
for p in prefixes:
    sub = anomalies[anomalies['trace_name'].str.startswith(p)]
    if not sub.empty:
        anomaly_count = int(sub.shape[0])
        earliest_ts = int(sub['timestamp'].min())
    else:
        anomaly_count = 0
        earliest_ts = pd.NA
    focused_rows.append({'trace_target': p, 'anomaly_count': anomaly_count, 'earliest_anomaly_ts': earliest_ts})

focused_summary = pd.DataFrame(focused_rows)

# Display compact outputs (variables)
anomalies_top20, agg_by_tracename, focused_summary

```
Out[5]:
```
Summary of trace anomaly analysis for the incident window (2021-03-10 15:00–15:30 Asia/Shanghai):

Key observations (top anomaly rows)
- Many anomalies are concentrated at the start of the window (timestamp 1615359600 → 2021-03-10 15:00:00 CST). Example anomalous rows at that timestamp include:
  - MG01: trace.from_Tomcat04.duration_p95 (value 4.1984 vs p95 4.16924)
  - Tomcat01–Tomcat04: multiple trace.from_root.* entries (duration_mean, duration_p95, row_count) where observed value equals the global p95/p5.
  - dockerA2 and dockerB1: trace.from_root.* entries at 15:00 flagged as anomalies.
- Top anomaly rows show many traces whose observed values equal their global p95 or p5 (i.e., pegged at historical extremes).

Aggregated anomalies by trace_name (top entries)
- Highest aggregated anomaly counts include:
  - trace.self.duration_p95 (dockerA1) — 8 anomalies (earliest 1615359720 → 2021-03-10 15:02:00)
  - trace.self.duration_p95 (dockerB2) — 7 anomalies (earliest 1615359960 → 2021-03-10 15:06:00)
  - trace.self.duration_p95 (dockerA2) — 6 anomalies (earliest 1615359720 → 2021-03-10 15:02:00)
  - Several trace.*.row_count and trace.*.duration_p95 entries across MG and docker nodes with 5 anomalies each (earliest timestamps around 15:03–15:06).
  - Multiple trace.from_IG01 / trace.from_IG02 entries on Tomcat nodes with 4 anomalies each (earliest timestamps ~15:01–15:08).

Focused summary for traces to Tomcat targets
- trace.to_Tomcat01: 24 anomaly points, earliest anomaly at 1615359720 → 2021-03-10 15:02:00 CST
- trace.to_Tomcat02: 24 anomaly points, earliest anomaly at 1615359660 → 2021-03-10 15:01:00 CST
- trace.to_Tomcat03: 24 anomaly points, earliest anomaly at 1615359720 → 2021-03-10 15:02:00 CST
- trace.to_Tomcat04: 24 anomaly points, earliest anomaly at 1615359720 → 2021-03-10 15:02:00 CST

Interpretation and next steps
- Anomalies affect traces both into and out of Tomcat instances and several docker/MG components. The repeated counts (24 anomalies for each trace.to_Tomcat0X) indicate persistent deviations for requests routed to Tomcats across the window.
- Many anomalous trace points are exactly at their global p95/p5 values (pegged at historical extremes), rather than isolated spikes—this suggests a sustained shift or bound (high tail or low floor) for those trace metrics during the incident.
- Recommended investigation priority:
  1. Inspect routing and Tomcat endpoints (Tomcat01–04) because all trace.to_Tomcat* have large anomaly counts.
  2. Check dockerA/B and MG components that show frequent trace.self.duration_p95 and row_count anomalies.
  3. Correlate these trace anomalies with service logs, deployments, or network changes around 15:00–15:02 CST to identify a common trigger.

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                        trace_name   timestamp      value        p95         p5
40       MG01  trace.from_Tomcat04.duration_p95  1615359600   4.198400   4.169240   1.129338
114  Tomcat01     trace.from_root.duration_mean  1615359600   0.002167   0.002167   0.002167
115  Tomcat01      trace.from_root.duration_p95  1615359600   0.009750   0.009750   0.009750
116  Tomcat01         trace.from_root.row_count  1615359600   6.000000   6.000000   6.000000
132  Tomcat02     trace.from_root.duration_mean  1615359600   0.003636   0.003636   0.003636
133  Tomcat02      trace.from_root.duration_p95  1615359600   0.020000   0.020000   0.020000
134  Tomcat02         trace.from_root.row_count  1615359600  11.000000  11.000000  11.000000
150  Tomcat03     trace.from_root.duration_mean  1615359600   0.000000   0.000000   0.000000
151  Tomcat03      trace.from_root.duration_p95  1615359600   0.000000   0.000000   0.000000
152  Tomcat03         trace.from_root.row_count  1615359600   2.000000   2.000000   2.000000
168  Tomcat04     trace.from_root.duration_mean  1615359600   0.000000   0.000000   0.000000
169  Tomcat04      trace.from_root.duration_p95  1615359600   0.000000   0.000000   0.000000
170  Tomcat04         trace.from_root.row_count  1615359600   6.000000   6.000000   6.000000
175  Tomcat04        trace.to_MG01.duration_p95  1615359600   4.198400   4.169240   1.129338
201  dockerA2     trace.from_root.duration_mean  1615359600   0.185909   0.185909   0.185909
202  dockerA2      trace.from_root.duration_p95  1615359600   1.017000   1.017000   1.017000
203  dockerA2         trace.from_root.row_count  1615359600  11.000000  11.000000  11.000000
219  dockerB1     trace.from_root.duration_mean  1615359600   0.139933   0.139933   0.139933
220  dockerB1      trace.from_root.duration_p95  1615359600   1.017000   1.017000   1.017000
221  dockerB1         trace.from_root.row_count  1615359600  89.000000  89.000000  89.000000,                            trace_name   cmdb_id  anomaly_count  earliest_anomaly_ts
137           trace.self.duration_p95  dockerA1              8           1615359720
140           trace.self.duration_p95  dockerB2              7           1615359960
138           trace.self.duration_p95  dockerA2              6           1615359720
76      trace.from_dockerA1.row_count      MG01              5           1615359780
242       trace.to_dockerB1.row_count      MG02              5           1615359840
173           trace.to_MG01.row_count  dockerA1              5           1615359780
81   trace.from_dockerA2.duration_p95      MG02              5           1615359960
190        trace.to_MG02.duration_p95  dockerA2              5           1615359960
46          trace.from_MG02.row_count  dockerB1              5           1615359840
9           trace.from_IG01.row_count  Tomcat02              4           1615359720
10          trace.from_IG01.row_count  Tomcat03              4           1615359720
11          trace.from_IG01.row_count  Tomcat04              4           1615359720
12      trace.from_IG02.duration_mean  Tomcat01              4           1615360080
13      trace.from_IG02.duration_mean  Tomcat02              4           1615360080
14      trace.from_IG02.duration_mean  Tomcat03              4           1615359960
15      trace.from_IG02.duration_mean  Tomcat04              4           1615360080
16       trace.from_IG02.duration_p95  Tomcat01              4           1615360080
17       trace.from_IG02.duration_p95  Tomcat02              4           1615360080
18       trace.from_IG02.duration_p95  Tomcat03              4           1615360080
1       trace.from_IG01.duration_mean  Tomcat02              4           1615359660,         trace_target  anomaly_count  earliest_anomaly_ts
0  trace.to_Tomcat01             24           1615359720
1  trace.to_Tomcat02             24           1615359660
2  trace.to_Tomcat03             24           1615359720
3  trace.to_Tomcat04             24           1615359720)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse in-memory metric_df if available, otherwise load from CSV
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone (UTC+8) per rules (used if converting timestamps; kept for consistency)
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the full series BEFORE filtering
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Select metric rows at the specific timestamp for the target components and join thresholds
target_components = ['Mysql01','Mysql02','Redis01','Redis02','Tomcat01','Tomcat02','Tomcat03','Tomcat04']
target_ts = 1615359600  # 2021-03-10 15:00:00 CST (Asia/Shanghai)

selected = metric_df[
    (metric_df['cmdb_id'].isin(target_components)) &
    (metric_df['timestamp'] == target_ts)
].copy()

# Merge computed thresholds (p95/p5) into the selected rows
selected_with_thresh = selected.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Compute flag per row: EQ_P5 (value == p5), GE_P95 (value >= p95), else NORMAL
# Use isclose for equality to handle float precision; handle NaNs safely
def determine_flag(row):
    v = row['value']
    p5 = row['p5']
    p95 = row['p95']
    # If thresholds missing, mark NORMAL
    if pd.isna(v) or pd.isna(p5) and pd.isna(p95):
        return 'NORMAL'
    # Check equality to p5 first
    if not pd.isna(p5) and np.isclose(v, p5, atol=1e-9, rtol=1e-6):
        return 'EQ_P5'
    # Then check >= p95
    if not pd.isna(p95) and v >= p95:
        return 'GE_P95'
    return 'NORMAL'

# Apply flag
selected_with_thresh['flag'] = selected_with_thresh.apply(determine_flag, axis=1)

# 4) Prepare final sorted result: sort by cmdb_id then kpi_name, limit top 200, but show top 50 in output
result_sorted = selected_with_thresh.sort_values(['cmdb_id', 'kpi_name'], ascending=[True, True])
result_top200 = result_sorted.head(200)

# Select and order output columns as requested
result_top200 = result_top200.loc[:, ['cmdb_id', 'kpi_name', 'timestamp', 'value', 'p95', 'p5', 'flag']]

# Prepare display: limit printed output to top 50 rows
result_top50 = result_top200.head(50)

# Display compact outputs (head 50 and shape of top200)
result_top50, result_top200.shape

```
Out[6]:
```
Summary of metrics at 2021-03-10 15:00:00 (Asia/Shanghai) for the eight target components:

- Scope
  - I selected 200 metric rows at timestamp 1615359600 (2021-03-10 15:00:00 CST) for Mysql01, Mysql02, Redis01, Redis02, Tomcat01–04 and joined each row with its global P95 and P5 computed from the full series.
  - The displayed sample shows the top 50 rows (sorted by cmdb_id, kpi_name). The full result set contains 200 rows.

- Key findings
  - Many MySQL metrics at this timestamp are exactly equal to their global P5 (flag = EQ_P5). Examples: Mysql-MySQL_3306_Aborted Clients, Aborted Connects, various Innodb reads/metrics are 0.0 and flagged EQ_P5.
  - Some metrics are flagged GE_P95 (value >= global P95). Example: Mysql-MySQL_3306_Innodb buffer pool write requests (value 24.8667 = p95) is flagged GE_P95.
  - Other metrics are in a normal range relative to their global thresholds (flag = NORMAL), e.g., Binlog Cache Use (value 2.6167 between p5 and p95).
  - Overall pattern: many metrics are pegged at historical low (P5), while a smaller subset are at/high beyond P95. This indicates a mix of suppressed activity for some KPIs (lots of zeros) and extreme high activity for a few KPIs at the same timestamp.

- Immediate interpretation and recommended next steps
  - Prioritize investigating MySQL nodes first (many KPIs at P5 and several at P95), then Redis and Tomcat, because the snapshot shows systemic deviations at the same timestamp.
  - For MySQL, inspect the zero-valued KPIs to confirm whether they indicate real drops (e.g., no activity) or measurement/collection issues, and examine the KPIs at/above P95 (e.g., Innodb write requests) for contention or overload.
  - Correlate this timestamp (2021-03-10 15:00 CST) with deployments, configuration changes, network events, or scheduled jobs that could explain the simultaneous shift across many metrics.
  - Check logs and traces for the same time window to identify root cause (locks, long queries, GC, network errors, etc.).

If you want, I can show:
- the full top-200 joined table (or specific components only), or
- a concise count of how many rows are EQ_P5 vs GE_P95 vs NORMAL across the 200 rows. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name   timestamp     value       p95        p5    flag
0   Mysql01                   Mysql-MySQL_3306_Aborted Clients  1615359600    0.0000    0.0000    0.0000   EQ_P5
1   Mysql01                  Mysql-MySQL_3306_Aborted Connects  1615359600    0.0000    0.0000    0.0000   EQ_P5
2   Mysql01             Mysql-MySQL_3306_Binlog Cache Disk Use  1615359600    0.0000    0.0000    0.0000   EQ_P5
3   Mysql01                  Mysql-MySQL_3306_Binlog Cache Use  1615359600    2.6167    2.6949    1.7000  NORMAL
4   Mysql01        Mysql-MySQL_3306_Binlog stmt cache disk use  1615359600    0.0000    0.0000    0.0000   EQ_P5
..      ...                                                ...         ...       ...       ...       ...     ...
45  Mysql01  Mysql-MySQL_3306_Innodb buffer pool read ahead...  1615359600    0.0000    0.0000    0.0000   EQ_P5
46  Mysql01  Mysql-MySQL_3306_Innodb buffer pool read requests  1615359600  460.4833  516.4167  230.4667  NORMAL
47  Mysql01          Mysql-MySQL_3306_Innodb buffer pool reads  1615359600    0.0000    0.0000    0.0000   EQ_P5
48  Mysql01      Mysql-MySQL_3306_Innodb buffer pool wait free  1615359600    0.0000    0.0000    0.0000   EQ_P5
49  Mysql01  Mysql-MySQL_3306_Innodb buffer pool write requ...  1615359600   24.8667   24.8667   15.6333  GE_P95

[50 rows x 7 columns], (200, 7))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import numpy as np
import pandas as pd
import pytz

# Reuse previously prepared snapshot at timestamp 1615359600
# Candidate variables from prior steps: selected_with_thresh, result_top200, result_top50
# Prefer selected_with_thresh (it was the full selection merged with thresholds)
if 'selected_with_thresh' in globals():
    snapshot = selected_with_thresh.copy()
elif 'result_top200' in globals():
    snapshot = result_top200.copy()
else:
    # fallback: reload metric and recompute thresholds then filter (should not be needed)
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = metric_df['timestamp'].astype(int)
    tz = pytz.timezone('Asia/Shanghai')
    thresholds = (
        metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
    )
    target_ts = 1615359600
    target_components = ['Mysql01','Mysql02','Redis01','Redis02','Tomcat01','Tomcat02','Tomcat03','Tomcat04']
    snapshot = metric_df[
        (metric_df['cmdb_id'].isin(target_components)) &
        (metric_df['timestamp'] == target_ts)
    ].merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Ensure we focus only on the requested 8 components
target_components = ['Mysql01','Mysql02','Redis01','Redis02','Tomcat01','Tomcat02','Tomcat03','Tomcat04']
snapshot = snapshot[snapshot['cmdb_id'].isin(target_components)].copy()

# Ensure numeric types
snapshot['value'] = pd.to_numeric(snapshot['value'], errors='coerce')
snapshot['p95'] = pd.to_numeric(snapshot['p95'], errors='coerce')
snapshot['p5'] = pd.to_numeric(snapshot['p5'], errors='coerce')

# Masks for conditions
eq_mask = np.isclose(snapshot['value'].fillna(np.nan), snapshot['p5'].fillna(np.nan), atol=1e-9, rtol=1e-6)
ge_mask = snapshot['value'] >= snapshot['p95']

snapshot['eq_p5'] = eq_mask
snapshot['ge_p95'] = ge_mask

# Relative deviations
denom_p95 = np.where(snapshot['p95'] == 0, 1.0, snapshot['p95'])
snapshot['rel_ge'] = (snapshot['value'] - snapshot['p95']) / denom_p95

denom_p5norm = np.maximum(snapshot['p5'], 1.0)
snapshot['rel_eq'] = (snapshot['p5'] - snapshot['value']) / denom_p5norm

# Group-level aggregates
grp = snapshot.groupby('cmdb_id')

summary = pd.DataFrame({
    'cmdb_id': grp.size().index
}).set_index('cmdb_id')

summary['n_kpis_sampled'] = grp.size()
summary['n_eq_p5'] = grp['eq_p5'].sum()
summary['n_ge_p95'] = grp['ge_p95'].sum()

summary['pct_eq_p5'] = (summary['n_eq_p5'] / summary['n_kpis_sampled']) * 100.0
summary['pct_ge_p95'] = (summary['n_ge_p95'] / summary['n_kpis_sampled']) * 100.0

# mean relative deviations (safe: if no rows matching, fill 0)
mean_rel_ge = snapshot[snapshot['ge_p95']].groupby('cmdb_id')['rel_ge'].mean().fillna(0.0)
mean_rel_eq = snapshot[snapshot['eq_p5']].groupby('cmdb_id')['rel_eq'].mean().fillna(0.0)

summary['mean_rel_deviation_ge_p95'] = mean_rel_ge
summary['mean_rel_deviation_eq_p5'] = mean_rel_eq

summary[['mean_rel_deviation_ge_p95','mean_rel_deviation_eq_p5']] = summary[['mean_rel_deviation_ge_p95','mean_rel_deviation_eq_p5']].fillna(0.0)

# overall severity score as specified
summary['overall_severity_score'] = (
    summary['pct_ge_p95'] * summary['mean_rel_deviation_ge_p95']
    + summary['pct_eq_p5'] * summary['mean_rel_deviation_eq_p5']
)

# Reset index for display and sort descending by overall_severity_score
summary = summary.reset_index().sort_values('overall_severity_score', ascending=False)

# Select requested columns and limit to the 8 components (already filtered)
output = summary.loc[:, [
    'cmdb_id', 'n_kpis_sampled', 'n_eq_p5', 'n_ge_p95',
    'pct_eq_p5', 'pct_ge_p95',
    'mean_rel_deviation_ge_p95', 'mean_rel_deviation_eq_p5',
    'overall_severity_score'
]]

# Keep output compact (<=20 rows). Show result.
output.head(20)

```
Out[7]:
```
summary = (
    "Per-component severity summary (snapshot at 2021-03-10 15:00 CST):\n\n"
    "Ranking (by overall_severity_score):\n"
    "1) Redis01 — highest severity (score 2.91). Sampled 17 KPIs; ~35% of KPIs equal P5 and ~35% >= P95. "
    "When >=P95 the mean relative excess is ~8.2% above P95. Investigate Redis01 first.\n\n"
    "2) Tomcat02 — second (score 1.48). Sampled 57 KPIs; ~66% EQ_P5 and ~63% GE_P95. Mean relative excess when >=P95 is ~2.3%. "
    "Persistent anomalies across many Tomcat02 KPIs warrant timely investigation.\n\n"
    "3) Mysql02 — third (score 0.69). Sampled 203 KPIs; ~60% EQ_P5 and ~54% GE_P95. Mean relative excess when >=P95 is ~1.3%. "
    "Large KPI count with many pegged at P5 or above P95 indicates broad MySQL disturbance.\n\n"
    "4) Mysql01 — fourth (score 0.15). Sampled 134 KPIs; ~63% EQ_P5 and ~69% GE_P95, but mean relative excess is small (~0.2%).\n\n"
    "Lower-scoring components (Tomcat04, Tomcat03, Redis02, Tomcat01) show many KPIs pegged at P5 or P95 too, but their relative deviations above P95 are very small, yielding low overall severity scores.\n\n"
    "Interpretation and recommended next steps:\n"
    "- Prioritize Redis01 and Tomcat02 for immediate investigation (highest severity). Then escalate to Mysql02 and Mysql01 due to large counts of affected KPIs.\n"
    "- The pattern combines many KPIs equal to global P5 (often zero) and a substantial subset >= P95; this suggests simultaneous suppression of some metrics and tail spikes in others at the same timestamp.\n"
    "- Correlate with logs, traces, deployments, and network events around 2021-03-10 15:00 CST to identify a common cause.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  n_kpis_sampled  n_eq_p5  n_ge_p95  pct_eq_p5  pct_ge_p95  mean_rel_deviation_ge_p95  mean_rel_deviation_eq_p5  overall_severity_score
2   Redis01              17        6         6  35.294118   35.294118                   0.082318                       0.0                2.905357
5  Tomcat02              57       38        36  66.666667   63.157895                   0.023452                       0.0                1.481177
1   Mysql02             203      122       110  60.098522   54.187192                   0.012743                       0.0                0.690499
0   Mysql01             134       85        93  63.432836   69.402985                   0.002137                       0.0                0.148347
7  Tomcat04              31       22        22  70.967742   70.967742                   0.000039                       0.0                0.002761
6  Tomcat03              53       35        35  66.037736   66.037736                   0.000016                       0.0                0.001060
3   Redis02              21       10         9  47.619048   42.857143                   0.000000                       0.0                0.000000
4  Tomcat01              20       14        14  70.000000   70.000000                   0.000000                       0.0                0.000000```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import re

# Reuse snapshot (selected_with_thresh) from memory if available
if 'selected_with_thresh' in globals():
    snapshot = selected_with_thresh.copy()
elif 'snapshot' in globals():
    snapshot = snapshot.copy()
elif 'result_top200' in globals():
    snapshot = result_top200.copy()
else:
    # Fallback: reload metric.csv and recompute thresholds then filter to timestamp 1615359600 for target components
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = metric_df['timestamp'].astype(int)
    thresholds = (
        metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
    )
    target_ts = 1615359600
    target_components_all = ['Mysql01','Mysql02','Redis01','Redis02','Tomcat01','Tomcat02','Tomcat03','Tomcat04']
    snapshot = (
        metric_df[
            (metric_df['cmdb_id'].isin(target_components_all)) &
            (metric_df['timestamp'] == target_ts)
        ]
        .merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    )

# Focus components
components = ['Redis01', 'Mysql02']

# Substring keywords (case-insensitive)
substrings = [
    'disk', 'io', 'fs', 'filesystem', 'cpu', 'mem', 'memory', 'swap',
    'tcp', 'network', 'conn', 'connections', 'innodb', 'innodb buffer',
    'evict', 'evictions', 'load'
]
pattern = '(' + '|'.join(re.escape(s) for s in substrings) + ')'

# Filter snapshot to the two components and KPI name matches
df_focus = snapshot[snapshot['cmdb_id'].isin(components)].copy()
df_focus['kpi_name_lower'] = df_focus['kpi_name'].astype(str).str.lower()
mask = df_focus['kpi_name_lower'].str.contains(pattern, case=False, regex=True, na=False)
df_res = df_focus[mask].copy()

# Compute numeric columns safely
df_res['value'] = pd.to_numeric(df_res['value'], errors='coerce')
df_res['p95'] = pd.to_numeric(df_res['p95'], errors='coerce')
df_res['p5'] = pd.to_numeric(df_res['p5'], errors='coerce')

# Flags: EQ_P5, GE_P95, else NORMAL
eq_mask = np.isclose(df_res['value'].fillna(np.nan), df_res['p5'].fillna(np.nan), atol=1e-9, rtol=1e-6)
ge_mask = df_res['value'] >= df_res['p95']
df_res['flag'] = np.where(eq_mask, 'EQ_P5', np.where(ge_mask, 'GE_P95', 'NORMAL'))

# Prepare output rows: select columns and sort, limit to top 100 (show top 50)
out_rows = df_res.loc[:, ['cmdb_id', 'kpi_name', 'timestamp', 'value', 'p95', 'p5', 'flag']] \
                 .sort_values(['cmdb_id', 'kpi_name'], ascending=[True, True]) \
                 .head(100)
out_rows_head50 = out_rows.head(50)

# Aggregated one-line counts per component
agg = (
    df_res
    .groupby('cmdb_id')
    .agg(
        n_rows=('kpi_name', 'size'),
        n_eq_p5=('flag', lambda s: (s == 'EQ_P5').sum()),
        n_ge_p95=('flag', lambda s: (s == 'GE_P95').sum())
    )
    .reset_index()
)

# Display compact results
out_rows_head50, agg

```
Out[8]:
```
Summary of resource-oriented KPI snapshot at 2021-03-10 15:00:00 CST for Redis01 and Mysql02:

What was selected
- I filtered metric rows for Redis01 and Mysql02 where kpi_name (case-insensitive) contains any of: disk, io, fs, filesystem, cpu, mem, memory, swap, tcp, network, conn, connections, innodb, innodb buffer, evict, evictions, load.
- Returned rows (top 50 shown) and a compact per-component aggregate.

Key counts (aggregate)
- Mysql02: 129 matching resource KPIs; 71 rows equal to global P5 (EQ_P5), 10 rows >= global P95 (GE_P95).
- Redis01: 14 matching resource KPIs; 3 EQ_P5, 2 GE_P95.

Notable patterns / examples
- Mysql02: many resource KPIs are exactly at their global P5 (often 0.0), e.g., Aborted Connects, Binlog Cache Disk Use, various Innodb/IO-related metrics — indicating either strongly suppressed activity or measurement zeros. Some MySQL KPIs are at/above P95 (fewer in number), while others (e.g., Questions) are within normal range relative to thresholds.
- Redis01: far fewer resource KPIs matched; a small number are at extremes (3 EQ_P5, 2 GE_P95), indicating some resource metrics also hit historic bounds but on a smaller scale than MySQL.

Interpretation and recommended next steps
- Mysql02 shows broad resource-related anomalies at this timestamp, dominated by many KPIs pegged to P5 (zeros) with a smaller subset at/above P95 — investigate MySQL collection/visibility and real activity (e.g., check if metrics are genuinely zero, instrumentation issues, or a service-level drop).
- For KPIs that are GE_P95, inspect for overload/latency/IO contention (focus on Innodb, write/read requests, connections).
- For Redis01, follow up on the few GE_P95 rows (possible high-load signs) and verify EQ_P5 rows similarly (are they expected zeros?).
- Correlate these findings with logs, traces, and recent deploys or network changes at 15:00 CST to find a common cause.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                     kpi_name   timestamp         value           p95            p5    flag
135  Mysql02            Mysql-MySQL_3306_Aborted Connects  1615359600  0.000000e+00  0.000000e+00  0.000000e+00   EQ_P5
136  Mysql02       Mysql-MySQL_3306_Binlog Cache Disk Use  1615359600  0.000000e+00  0.000000e+00  0.000000e+00   EQ_P5
138  Mysql02  Mysql-MySQL_3306_Binlog stmt cache disk use  1615359600  0.000000e+00  0.000000e+00  0.000000e+00   EQ_P5
146  Mysql02                    Mysql-MySQL_3306_Com Load  1615359600  0.000000e+00  0.000000e+00  0.000000e+00   EQ_P5
152  Mysql02                 Mysql-MySQL_3306_Connections  1615359600  5.000000e-02  5.080000e-02  4.920000e-02  NORMAL
..       ...                                          ...         ...           ...           ...           ...     ...
215  Mysql02              Mysql-MySQL_3306_MaxConnections  1615359600  2.000000e+03  2.000000e+03  2.000000e+03   EQ_P5
221  Mysql02    Mysql-MySQL_3306_Opened table definitions  1615359600  0.000000e+00  0.000000e+00  0.000000e+00   EQ_P5
223  Mysql02          Mysql-MySQL_3306_Qcache Free Memory  1615359600  1.031832e+06  1.031832e+06  1.031832e+06   EQ_P5
226  Mysql02        Mysql-MySQL_3306_Qcache Lowmem Prunes  1615359600  0.000000e+00  0.000000e+00  0.000000e+00   EQ_P5
230  Mysql02                   Mysql-MySQL_3306_Questions  1615359600  4.985833e+02  6.850167e+02  3.187833e+02  NORMAL

[50 rows x 7 columns],    cmdb_id  n_rows  n_eq_p5  n_ge_p95
0  Mysql02     129       71        10
1  Redis01      14        3         2)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
import re

# Reuse existing metric_df if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone per rules (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the full series (before filtering)
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Select rows at the specific timestamp for the target components
target_components = ['Mysql01', 'Mysql02', 'Redis01', 'Redis02']
target_ts = 1615359600  # 2021-03-10 15:00:00 CST (Asia/Shanghai)

selected = (
    metric_df[
        (metric_df['cmdb_id'].isin(target_components)) &
        (metric_df['timestamp'] == target_ts)
    ]
    .merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    .copy()
)

# 3) Filter kpi_name by keywords (case-insensitive)
keywords = [
    'fs', 'filesystem', 'disk', 'io', 'read', 'write', 'throughput', 'iops',
    'cpu', 'load', 'mem', 'memory', 'swap', 'tcp', 'network', 'conn', 'connections',
    'innodb', 'buffer pool', 'evict', 'threads', 'maxconnections', 'created tmp',
    'opened table', 'queries', 'questions', 'evictions'
]
# Build regex (escape and join). Use word fragments to match broad KPI naming.
pattern = '(' + '|'.join(re.escape(k) for k in keywords) + ')'
selected['kpi_name_lower'] = selected['kpi_name'].astype(str).str.lower()
mask = selected['kpi_name_lower'].str.contains(pattern, case=False, regex=True, na=False)
res = selected[mask].copy()

# 4) Compute flag: EQ_P5 (value == p5), GE_P95 (value >= p95), else NORMAL
# Use isclose for equality; handle NaNs
res['value'] = pd.to_numeric(res['value'], errors='coerce')
res['p95'] = pd.to_numeric(res['p95'], errors='coerce')
res['p5'] = pd.to_numeric(res['p5'], errors='coerce')

eq_mask = np.isclose(res['value'].fillna(np.nan), res['p5'].fillna(np.nan), atol=1e-9, rtol=1e-6)
ge_mask = (res['value'] >= res['p95']) & (~res['p95'].isna())

res['flag'] = np.where(eq_mask, 'EQ_P5', np.where(ge_mask, 'GE_P95', 'NORMAL'))

# Relative severity for GE_P95: (value - p95) / max(p95,1)
denom = np.maximum(res['p95'].fillna(0.0), 1.0)
res['rel_over_p95'] = (res['value'].fillna(0.0) - res['p95'].fillna(0.0)) / denom

# 5) Prepare top 100 rows sorted by cmdb_id then kpi_name (compact columns)
matched_top100 = (
    res
    .sort_values(['cmdb_id', 'kpi_name'], ascending=[True, True])
    .loc[:, ['cmdb_id', 'kpi_name', 'timestamp', 'value', 'p95', 'p5', 'flag']]
    .head(100)
)

# 6) Per-component resource-summary
def top_ge_p95_kpis(df_grp, topn=5):
    ge = df_grp[df_grp['flag'] == 'GE_P95'].copy()
    if ge.empty:
        return ''
    topk = ge.sort_values('rel_over_p95', ascending=False).head(topn)
    # join kpi names with relative percent for context
    items = [
        f"{row.kpi_name}({row.rel_over_p95:.3f})"
        for _, row in topk.iterrows()
    ]
    return '; '.join(items)

agg = (
    res
    .groupby('cmdb_id')
    .agg(
        n_resource_kpis_sampled=('kpi_name', 'size'),
        n_eq_p5=('flag', lambda s: (s == 'EQ_P5').sum()),
        n_ge_p95=('flag', lambda s: (s == 'GE_P95').sum())
    )
    .reset_index()
)

# add list of up to 5 most extreme GE_P95 KPI names per component
extreme_lists = []
for comp in agg['cmdb_id']:
    df_comp = res[res['cmdb_id'] == comp]
    extreme_lists.append(top_ge_p95_kpis(df_comp, topn=5))

agg['top_ge_p95_kpis'] = extreme_lists

# Keep output compact: select columns order
comp_summary = agg.loc[:, ['cmdb_id', 'n_resource_kpis_sampled', 'n_eq_p5', 'n_ge_p95', 'top_ge_p95_kpis']]

# Final outputs: matched rows (top100) and compact per-component summary (<=20 rows)
matched_top100, comp_summary

```
Out[9]:
```
Compact findings from the 2021-03-10 15:00:00 CST metric snapshot for resource-oriented KPIs:

Per-component counts (resource KPIs matching your keywords)
- Mysql01: 85 KPIs sampled; 49 EQ_P5; 9 GE_P95.
- Mysql02: 151 KPIs sampled; 84 EQ_P5; 10 GE_P95.
- Redis01: 14 KPIs sampled; 3 EQ_P5; 2 GE_P95.
- Redis02: 18 KPIs sampled; 8 EQ_P5; 1 GE_P95.

Example most-extreme GE_P95 KPIs (up to 5 per component, truncated names shown)
- Mysql01 (examples): Mysql-MySQL_3306_Innodb buffer pool pages flus...
- Mysql02 (examples): Mysql-MySQL_3306_Innodb dblwr writes(...); (and others including high relative deviations)
- Redis01 (examples): OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...
- Redis02 (examples): OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem(...)

Key patterns and interpretation
- Many MySQL resource KPIs are exactly at their global P5 (a large number of zeros) — this is the dominant pattern for both Mysql01 and Mysql02. That suggests either genuine suppressed activity at that timestamp or a collection/visibility issue causing zero readings.
- A smaller but important subset of KPIs are >= P95 (GE_P95). For MySQL these include Innodb-related IO/write metrics and some connection-related KPIs — these indicate pockets of high activity or contention despite many other metrics being low.
- Redis shows far fewer resource metrics overall; some extremes exist (a couple GE_P95 rows and a few EQ_P5 rows) but on a smaller scale than MySQL.

Recommended next steps
- Investigate MySQL nodes first:
  - Verify whether the many zero-valued KPIs are real (service idle) or an instrumentation/collection problem.
  - Inspect Innodb IO/dblwr, buffer pool, write/read IOPS, and connection counts for contention or overload where GE_P95 is flagged.
- Check Redis nodes for the handful of GE_P95 indicators (network/connection or memory items) to rule out cache-layer impact.
- Correlate these metric anomalies with logs, traces, and any deploys or network changes around 2021-03-10 15:00 CST to identify a common root cause.

If you want, I can: (a) print the full matched row table (up to 100 rows), or (b) list the exact GE_P95 KPI names and their relative over-P95 values for each component. Which would you like?

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                     kpi_name   timestamp      value        p95        p5    flag
1    Mysql01            Mysql-MySQL_3306_Aborted Connects  1615359600     0.0000     0.0000    0.0000   EQ_P5
2    Mysql01       Mysql-MySQL_3306_Binlog Cache Disk Use  1615359600     0.0000     0.0000    0.0000   EQ_P5
4    Mysql01  Mysql-MySQL_3306_Binlog stmt cache disk use  1615359600     0.0000     0.0000    0.0000   EQ_P5
12   Mysql01                    Mysql-MySQL_3306_Com Load  1615359600     0.0000     0.0000    0.0000   EQ_P5
18   Mysql01                 Mysql-MySQL_3306_Connections  1615359600     0.0500     0.0508    0.0492  NORMAL
..       ...                                          ...         ...        ...        ...       ...     ...
161  Mysql02           Mysql-MySQL_3306_Handler Read Next  1615359600     0.0000     0.0000    0.0000   EQ_P5
162  Mysql02           Mysql-MySQL_3306_Handler Read Prev  1615359600  1004.2333  2430.9333  424.8689  NORMAL
163  Mysql02            Mysql-MySQL_3306_Handler Read Rnd  1615359600     0.0000     0.0000    0.0000   EQ_P5
164  Mysql02       Mysql-MySQL_3306_Handler Read Rnd Next  1615359600    28.9167    29.4407   28.4426  NORMAL
169  Mysql02               Mysql-MySQL_3306_Handler Write  1615359600   204.4833   253.8333  139.1333  NORMAL

[100 rows x 7 columns],    cmdb_id  n_resource_kpis_sampled  n_eq_p5  n_ge_p95                                    top_ge_p95_kpis
0  Mysql01                       85       49         9  Mysql-MySQL_3306_Innodb buffer pool pages flus...
1  Mysql02                      151       84        10  Mysql-MySQL_3306_Innodb dblwr writes(1.402); M...
2  Redis01                       14        3         2  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...
3  Redis02                       18        8         1    OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem(0.000))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import numpy as np
import pandas as pd
import pytz
import re

# Reuse the in-memory snapshot that was previously created: selected_with_thresh
# Fallback: if not present, raise error
if 'selected_with_thresh' in globals():
    snapshot = selected_with_thresh.copy()
else:
    raise RuntimeError("Expected in-memory DataFrame 'selected_with_thresh' not found.")

# Ensure numeric columns
snapshot['timestamp'] = snapshot['timestamp'].astype(int)
snapshot['value'] = pd.to_numeric(snapshot['value'], errors='coerce')
snapshot['p95'] = pd.to_numeric(snapshot['p95'], errors='coerce')
snapshot['p5'] = pd.to_numeric(snapshot['p5'], errors='coerce')

# Recompute flag to be safe: EQ_P5 if value == p5 (isclose), GE_P95 if value >= p95, else NORMAL
eq_mask = np.isclose(snapshot['value'].fillna(np.nan), snapshot['p5'].fillna(np.nan), atol=1e-9, rtol=1e-6)
ge_mask = (snapshot['value'] >= snapshot['p95']) & (~snapshot['p95'].isna())
snapshot['flag'] = np.where(eq_mask, 'EQ_P5', np.where(ge_mask, 'GE_P95', 'NORMAL'))

# Filter for the requested components and timestamp
components = ['Mysql02', 'Redis01']
target_ts = 1615359600
df_focus = snapshot[(snapshot['cmdb_id'].isin(components)) & (snapshot['timestamp'] == target_ts)].copy()

# Compute rel_dev = (value - p95) / max(p95, 1)
denom = np.maximum(df_focus['p95'].fillna(0.0), 1.0)
df_focus['rel_dev'] = (df_focus['value'].fillna(0.0) - df_focus['p95'].fillna(0.0)) / denom

# Select GE_P95 rows per component
ge_rows = df_focus[df_focus['flag'] == 'GE_P95'].copy()

# Prepare per-component top-N results
mysql_top10 = ge_rows[ge_rows['cmdb_id'] == 'Mysql02'].sort_values('rel_dev', ascending=False).loc[:, ['cmdb_id','kpi_name','value','p95','p5','rel_dev']].head(10)
redis_top10 = ge_rows[ge_rows['cmdb_id'] == 'Redis01'].sort_values('rel_dev', ascending=False).loc[:, ['cmdb_id','kpi_name','value','p95','p5','rel_dev']].head(10)

# Determine dominant resource category per component based on top GE_P95 KPI names
# Category mapping by keyword presence (case-insensitive)
category_keywords = {
    'disk I/O': ['io','read','write','iops','throughput','innodb','dblwr','buffer pool','innodb buffer','flush','write requests','reads','writes'],
    'disk space': ['fs','filesystem','disk space','usedspace','fsusedspace','fscapacity','fsavailable'],
    'high CPU': ['cpu','load','cpuload','cpusys','cpuuser'],
    'high memory': ['mem','memory','swap','memfree','memtotal','rss','cache','buffer pool pages'],
    'network latency/packet loss': ['tcp','network','conn','connections','packet','tcp-','totaltcp','tcpconn'],
    'other': []
}

def categorize_kpi(name):
    n = name.lower()
    for cat, keys in category_keywords.items():
        for k in keys:
            if k in n:
                return cat
    return 'other'

comp_summaries = []
for comp in components:
    comp_ge = ge_rows[ge_rows['cmdb_id'] == comp].copy()
    n_rows = int(comp_ge.shape[0])
    if n_rows == 0:
        dominant = 'none'
    else:
        # categorize each GE_P95 KPI and pick the most common category
        comp_ge['category'] = comp_ge['kpi_name'].astype(str).apply(categorize_kpi)
        cat_counts = comp_ge['category'].value_counts()
        dominant = cat_counts.idxmax()
    # also prepare top 5 most extreme GE_P95 KPI names by rel_dev
    top5_names = comp_ge.sort_values('rel_dev', ascending=False).head(5)['kpi_name'].tolist()
    comp_summaries.append({
        'cmdb_id': comp,
        'n_ge_p95_rows': n_rows,
        'dominant_category': dominant,
        'top5_ge_p95_kpis': '; '.join(top5_names)
    })

comp_summary_df = pd.DataFrame(comp_summaries).loc[:, ['cmdb_id','n_ge_p95_rows','dominant_category','top5_ge_p95_kpis']]

# Final compact outputs: mysql_top10, redis_top10, comp_summary_df
mysql_top10, redis_top10, comp_summary_df

```
Out[10]:
```
Summary of GE_P95 resource anomalies at 2021-03-10 15:00:00 CST (components Mysql02, Redis01):

Mysql02 (10 GE_P95 rows)
- Top GE_P95 by relative deviation:
  - Mysql-MySQL_3306_Innodb dblwr writes: value=4.60, p95=1.9153, rel_dev ≈ 1.402 (≈140% above p95) — largest signal.
  - Several other GE_P95 rows are filesystem / Innodb / capacity metrics that match or barely exceed p95 (many rel_dev = 0 because value == p95).
- Dominant resource category (by top GE_P95 KPIs returned): disk space (note: the dblwr writes KPI also implies disk I/O activity for InnoDB).
- Implication: Mysql02 shows a notable InnoDB double-write / disk-I/O spike (highest relative deviation) plus multiple filesystem/buffer-related KPIs at/past their p95 — investigate disk I/O path, InnoDB write activity and filesystem health/usage.

Redis01 (2 GE_P95 rows)
- Top GE_P95 by relative deviation:
  - OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum: value=1104, p95=739, rel_dev ≈ 0.494 (~49% above p95) — largest signal for Redis01.
  - A zabbix-agent process metric flagged GE_P95 but with rel_dev ≈ 0.
- Dominant resource category: network latency/packet loss (based on the high TCP connection count).
- Implication: Redis01 shows unusually high TCP connection counts at this timestamp — investigate network/connectivity and client connection behavior.

Actionable next steps (concise)
- For Mysql02: prioritize disk I/O / InnoDB investigation (long or heavy dblwr writes, fsyncs, buffer pool activity), check underlying storage (I/O latency, throughput, queueing) and queries causing writes.
- For Redis01: inspect client connection patterns and network metrics (socket counts, connection spikes, possible retransmits or load balancer issues).

If you want, I can print the exact top-10 GE_P95 rows (with rel_dev) for each component again.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name      value        p95           p5   rel_dev
193  Mysql02               Mysql-MySQL_3306_Innodb dblwr writes     4.6000     1.9153     0.000000  1.401713
174  Mysql02     Mysql-MySQL_3306_Innodb buffer pool pages free  8193.0000  8193.0000  8176.000000  0.000000
185  Mysql02        Mysql-MySQL_3306_Innodb data pending fsyncs     1.0000     1.0000     0.000000  0.000000
199  Mysql02      Mysql-MySQL_3306_Innodb os log pending fsyncs     1.0000     1.0000     0.000000  0.000000
270  Mysql02        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity    25.2434    25.2434    25.243358  0.000000
271  Mysql02  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...     1.1715     1.1715     1.171498  0.000000
274  Mysql02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity     0.4309     0.4309     0.430851  0.000000
275  Mysql02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...     0.0104     0.0104     0.010376  0.000000
291  Mysql02  OSLinux-OSLinux_FILESYSTEM_-mysqldata_FSInodeU...     0.0341     0.0341     0.034058  0.000000
299  Mysql02  OSLinux-OSLinux_FILESYSTEM_-usr-openv_FSInodeU...     0.0017     0.0017     0.001678  0.000000,      cmdb_id                                           kpi_name   value    p95     p5   rel_dev
346  Redis01    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1104.0  739.0  603.0  0.493911
347  Redis01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     0.4    0.4    0.3  0.000000,    cmdb_id  n_ge_p95_rows            dominant_category                                   top5_ge_p95_kpis
0  Mysql02             10                   disk space  Mysql-MySQL_3306_Innodb dblwr writes; Mysql-My...
1  Redis01              2  network latency/packet loss  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse in-memory 'anomalies' if available; otherwise reconstruct from metric.csv
tz = pytz.timezone('Asia/Shanghai')

if 'anomalies' in globals():
    anom_df = anomalies.copy()
else:
    # Reconstruct thresholds and window anomalies (fallback)
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = metric_df['timestamp'].astype(int)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')
    thresholds = (
        metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
    )
    metric_with_thresh = metric_df.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
    start = pd.Timestamp('2021-03-10 15:00:00').tz_localize(tz)
    end = pd.Timestamp('2021-03-10 15:30:00').tz_localize(tz)
    metric_with_thresh['dt'] = pd.to_datetime(metric_with_thresh['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    window_df = metric_with_thresh[(metric_with_thresh['dt'] >= start) & (metric_with_thresh['dt'] <= end)].copy()
    window_df['is_anomaly'] = (window_df['value'] >= window_df['p95']) | (window_df['value'] <= window_df['p5'])
    anom_df = window_df[window_df['is_anomaly']].copy()

# Focus components
components = ['Mysql02', 'Redis01']

rows = []
for comp in components:
    comp_ge = anom_df[(anom_df['cmdb_id'] == comp) & (anom_df['value'] >= anom_df['p95'])].copy()
    if comp_ge.empty:
        # No GE_P95 found: produce a null/placeholder row
        rows.append({
            'cmdb_id': comp,
            'kpi_name': pd.NA,
            'timestamp': pd.NA,
            'value': pd.NA,
            'p95': pd.NA,
            'p5': pd.NA,
            'rel_dev': pd.NA
        })
        continue
    # Earliest timestamp when any GE_P95 occurred
    earliest_ts = int(comp_ge['timestamp'].min())
    # Select GE_P95 rows at that earliest timestamp
    at_earliest = comp_ge[comp_ge['timestamp'] == earliest_ts].copy()
    # Compute rel_dev using denominator = max(p95, 1)
    denom = at_earliest['p95'].fillna(0.0).apply(lambda x: max(x, 1.0))
    at_earliest['rel_dev'] = (at_earliest['value'].fillna(0.0) - at_earliest['p95'].fillna(0.0)) / denom
    # Choose the row with largest rel_dev
    best = at_earliest.sort_values('rel_dev', ascending=False).iloc[0]
    rows.append({
        'cmdb_id': comp,
        'kpi_name': best['kpi_name'],
        'timestamp': int(best['timestamp']),
        'value': float(best['value']) if pd.notna(best['value']) else pd.NA,
        'p95': float(best['p95']) if pd.notna(best['p95']) else pd.NA,
        'p5': float(best['p5']) if pd.notna(best['p5']) else pd.NA,
        'rel_dev': float(best['rel_dev']) if pd.notna(best['rel_dev']) else pd.NA
    })

result = pd.DataFrame(rows)
result = result.sort_values('cmdb_id').reset_index(drop=True)

result

```
Out[11]:
```
Result summary (compact):

- For both Mysql02 and Redis01, no GE_P95 anomaly row was found in the in-memory incident-window anomaly set. The query returned no earliest GE_P95 timestamp or supporting KPI row for either component.

Possible reasons / quick checks:
- There genuinely were no value >= global P95 anomalies for these two components inside the incident window.
- Or the GE_P95 events exist in the full metric snapshot at a single timestamp (e.g., 1615359600) but were classified differently in the in-memory anomalies (e.g., EQ_P5 if value == p5) or p95 values are missing, causing no GE_P95 matches.
- Another possibility is that GE_P95 occurred at timestamps outside the filtered incident window.

If you want, I can immediately re-run a precise scan (using the stored full-series thresholds joined to metrics) to:
- explicitly search the incident window (2021-03-10 15:00–15:30 CST) for any rows where value >= p95 for Mysql02 and Redis01 and return the earliest such row with the largest relative deviation — I can run that now if you confirm.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id kpi_name timestamp value   p95    p5 rel_dev
0  Mysql02     <NA>      <NA>  <NA>  <NA>  <NA>    <NA>
1  Redis01     <NA>      <NA>  <NA>  <NA>  <NA>    <NA>```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing metric_df if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')

# Ensure types
metric_df['timestamp'] = metric_df['timestamp'].astype(int)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Timezone (UTC+8) per rules (used only if needed)
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using the full series (must be done BEFORE filtering)
if 'thresholds' in globals():
    thresholds_full = thresholds.copy()
else:
    thresholds_full = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(p95=lambda x: x.quantile(0.95), p5=lambda x: x.quantile(0.05))
        .reset_index()
    )

# 2) Filter metric rows to the incident window timestamps >=1615359600 and <=1615361400
start_ts = 1615359600
end_ts = 1615361400
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Merge thresholds into the window rows
window_with_thresh = window_df.merge(thresholds_full, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Find all rows in that window where value >= p95
# Ensure numeric p95
window_with_thresh['p95'] = pd.to_numeric(window_with_thresh['p95'], errors='coerce')
window_with_thresh['p5'] = pd.to_numeric(window_with_thresh['p5'], errors='coerce')

ge_mask = (window_with_thresh['value'] >= window_with_thresh['p95']) & (~window_with_thresh['p95'].isna())
ge_rows = window_with_thresh[ge_mask].copy()

# 4) For each component, compute required summaries and select the KPI row at earliest timestamp with largest rel_dev
components = ['Mysql02', 'Redis01']

rows = []
for comp in components:
    comp_ge = ge_rows[ge_rows['cmdb_id'] == comp].copy()
    total_count = int(comp_ge.shape[0])
    if total_count == 0:
        rows.append({
            'cmdb_id': comp,
            'count_ge_p95': 0,
            'earliest_ts': pd.NA,
            'kpi_name': pd.NA,
            'timestamp': pd.NA,
            'value': pd.NA,
            'p95': pd.NA,
            'p5': pd.NA,
            'rel_dev': pd.NA
        })
        continue
    earliest_ts = int(comp_ge['timestamp'].min())
    at_earliest = comp_ge[comp_ge['timestamp'] == earliest_ts].copy()
    # compute rel_dev using denominator = max(p95,1)
    denom = at_earliest['p95'].fillna(0.0).apply(lambda x: max(x, 1.0))
    at_earliest['rel_dev'] = (at_earliest['value'].fillna(0.0) - at_earliest['p95'].fillna(0.0)) / denom
    # select row with largest rel_dev (if ties, take first)
    best = at_earliest.sort_values('rel_dev', ascending=False).iloc[0]
    rows.append({
        'cmdb_id': comp,
        'count_ge_p95': total_count,
        'earliest_ts': earliest_ts,
        'kpi_name': best['kpi_name'],
        'timestamp': int(best['timestamp']),
        'value': float(best['value']) if pd.notna(best['value']) else pd.NA,
        'p95': float(best['p95']) if pd.notna(best['p95']) else pd.NA,
        'p5': float(best['p5']) if pd.notna(best['p5']) else pd.NA,
        'rel_dev': float(best['rel_dev']) if pd.notna(best['rel_dev']) else pd.NA
    })

output_df = pd.DataFrame(rows).sort_values('cmdb_id').reset_index(drop=True)

# Compact display: exactly two rows, requested columns
output_df.loc[:, ['cmdb_id', 'count_ge_p95', 'earliest_ts', 'kpi_name', 'timestamp', 'value', 'p95', 'p5', 'rel_dev']]

```
Out[12]:
```
Compact results and interpretation:

- Mysql02
  - Total GE_P95 rows in the incident window: 2326
  - Earliest GE_P95 timestamp: 1615359600 (2021-03-10 15:00:00 CST)
  - Representative KPI at that earliest time with largest relative deviation:
    - kpi_name: Mysql-MySQL_3306_Innodb dblwr writes
    - timestamp: 1615359600
    - value = 4.6, p95 = 1.9153, p5 = 0.0
    - rel_dev = (value - p95)/max(p95,1) ≈ 1.4017 (≈140% above p95)
  - Interpretation: Massive number of GE_P95 events and the top signal point to InnoDB double-write / disk I/O activity. Investigate disk I/O subsystem, InnoDB write patterns, fsyncs, and queries causing heavy writes.

- Redis01
  - Total GE_P95 rows in the incident window: 1301
  - Earliest GE_P95 timestamp: 1615359600 (2021-03-10 15:00:00 CST)
  - Representative KPI at that earliest time with largest relative deviation:
    - kpi_name: OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum
    - timestamp: 1615359600
    - value = 1104.0, p95 = 739.0, p5 = 603.0
    - rel_dev ≈ 0.4939 (≈49% above p95)
  - Interpretation: Large spike in TCP connection count. Investigate client connection behavior, load balancer or network issues, and possible connection storms.

Recommended next steps (concise)
- Prioritize Mysql02 disk/I/O and InnoDB investigations; correlate with slow queries, fsyncs, and storage metrics.
- For Redis01, check client connection patterns, network health, and any upstream changes around 15:00 CST.
- Correlate these findings with logs, traces, and deployments at the 15:00 timestamp to identify a common root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  count_ge_p95  earliest_ts                                         kpi_name   timestamp   value       p95     p5   rel_dev
0  Mysql02          2326   1615359600             Mysql-MySQL_3306_Innodb dblwr writes  1615359600     4.6    1.9153    0.0  1.401713
1  Redis01          1301   1615359600  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum  1615359600  1104.0  739.0000  603.0  0.493911```
```